The data in this project is a compilation of the monthly **output** of crude oil and condensate liquid from the major oil terminals/streams in Nigeria between January 2024 and June 2026.

The data was initially intended to be a compilation of crude oil production by field across Nigeria, however, gaining access to field-level data requires a paid license, but the terminal-level data has been made publicly available by the NUPRC.

This project is built to answer three questions:
1) Does each terminal show a downward, upward or stable trend in output, and how much does this vary from terminal to terminal?
2) To what degree does volatility exist in the output of each terminal on a month to month basis?
3) Do the top 20% of the terminals in the dataset account for 80% of the total output in compliance with the "Pareto principle"?

In [2]:
import pandas as pd
import numpy as np

In [3]:
df_2024 = pd.read_csv("../data/raw/NUPRC_2024_production_raw.csv")
df_2025 = pd.read_csv("../data/raw/NUPRC_2025_production_raw.csv")
df_2026 = pd.read_csv("../data/raw/NUPRC_2026_production_raw.csv")

In [4]:
df_2024.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,6351778.0,4604102.0,4260566.0,4158173.0,3605864.0,4536573.0,4584388.0,4894497.0,5589773.0,5765964.0,7051541.0,7274190.0
1,BONNY,Condensate,587495.0,583490.0,627057.0,543214.0,640026.0,568474.0,564610.0,584706.0,511021.0,494915.0,501329.0,509570.0
2,BONNY,Blend Total,6939273.0,5187592.0,4887623.0,4701387.0,4245890.0,5105047.0,5148998.0,5479203.0,6100794.0,6260879.0,7552870.0,7783759.0
3,BRASS,Crude Oil,735680.0,617189.0,686188.0,647053.0,635929.0,648849.0,664387.0,718104.0,842900.0,850352.0,775490.0,905544.0
4,BRASS,Condensate,160901.0,135498.0,158529.0,140724.0,153167.0,144677.0,148961.0,161633.0,201979.0,201451.0,179148.0,180417.0


In [5]:
df_2025.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,7570173.0,5892185.0,7360738.0,7079233.0,6673050.0,6625912.0,7567542.0,5850190.0,5924385.0,7535315.0,7751499.0,7884015.0
1,BONNY,Condensate,574015.0,475882.0,290820.0,326253.0,423241.0,542024.0,507637.0,418278.0,393815.0,399509.0,419547.0,406781.0
2,BONNY,Blend Total,8144187.0,6368067.0,7651558.0,7405486.0,7096291.0,7167936.0,8075179.0,6268468.0,6318200.0,7934823.0,8171046.0,8290796.0
3,BRASS,Crude Oil,860312.0,726912.0,937437.0,619261.0,857711.0,710231.0,820821.0,846393.0,935010.0,946439.0,981254.0,914165.0
4,BRASS,Condensate,190629.0,149116.0,180191.0,127826.0,175028.0,167744.0,294502.0,309192.0,217677.0,220768.0,225872.0,265032.0


In [6]:
df_2026.head()

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
0,BONNY,Crude Oil,232.30,262.88,258.00,278.68,277.04,301.93
1,BONNY,Condensate,12.57,13.18,13.78,16.43,16.83,16.34
2,BONNY,Blend Total,244.87,276.05,271.77,295.10,293.88,318.28
3,BRASS,Crude Oil,30.36,34.41,35.09,32.99,33.77,33.18
4,BRASS,Condensate,9.51,10.04,9.37,8.52,9.77,8.67


The values in the 2026 dataset are significantly lower than those of 2024 and 2024. This is because it is measured in thousand barells per day, unlike the data in 2024 and 2025 which is measured as the total number of barells produced. The unit conversion would be done in the cleaning phase.

In [7]:
print(df_2024.shape)
print(df_2025.shape)
print(df_2026.shape)

(56, 14)
(56, 14)
(54, 8)


In [8]:
# Confirms uniformity in Terminal names across datasets 2024 and 2025
df_2024.loc[:, "Terminal/Stream"] == df_2025.loc[:, "Terminal/Stream"]

0      True
1      True
2      True
3      True
4      True
5      True
6      True
7      True
8      True
9      True
10     True
11     True
12     True
13     True
14     True
15     True
16     True
17     True
18     True
19     True
20     True
21     True
22     True
23     True
24     True
25     True
26    False
27     True
28     True
29     True
30     True
31     True
32     True
33     True
34     True
35     True
36     True
37     True
38    False
39     True
40     True
41     True
42     True
43     True
44     True
45     True
46     True
47     True
48     True
49     True
50     True
51     True
52     True
53     True
54     True
55     True
Name: Terminal/Stream, dtype: bool

In [9]:
# Extracts the names of the different Terminals identified from the previous code
print(f"2024 Index 26 name: {df_2024.iloc[26, 0]}, 2025 Index 26 name: {df_2025.iloc[26, 0]}")
print(f"2024 Index 38 name: {df_2024.iloc[38, 0]}, 2025 Index 38 name: {df_2025.iloc[38, 0]}")

2024 Index 26 name: OTAKPIPO (Ex Ima Terminal), 2025 Index 26 name: OTAKPIPO
2024 Index 38 name: OYO, 2025 Index 38 name: OYO / OBODO


Comparing terminal names between 2024 and 2025: two mismatches found — 'OTAKPIPO (Ex Ima Terminal)' (2024) vs 'OTAKPIPO' (2025), and 'OYO' (2024) vs 'OYO / OBODO' (2025, likely reflecting a new grade addition). Same entity in both cases. Will standardize in cleaning, once I've checked whether 2026 introduces further variants.

In [10]:
#create a list of the Terminals/Streams in 2025 dataset
terminal_list_2025 = list(df_2025["Terminal/Stream"])
#create a list of the Terminals/Streams in 2026 dataset
terminal_list_2026 = list(df_2026["Terminal/Stream"])

#check which Terminals were removed from 2025 in 2026
removed_from_2025 = [terminal for terminal in terminal_list_2025 if terminal not in terminal_list_2026]
#check which Terminals were added in 2026 that were not in 2025
added_in_2026 = [terminal for terminal in terminal_list_2026 if terminal not in terminal_list_2025]
print(f"Terminals/Streams removed from 2025 in 2026: {removed_from_2025}")
print(f"Terminals/Streams added in 2026: {added_in_2026}")


Terminals/Streams removed from 2025 in 2026: ['ASARAMATORU (Ex Ima Terminal)', 'ANAMBRA BASIN', 'UKPOKITI']
Terminals/Streams added in 2026: ['CAWTHORNE']


As seen from the shape of the three datasets, data from the year 2026 has 2 fewer records(rows) than those of 2025 and 2024, as well as 6 fewer fields(columns).
The difference in fields is due to the fact that the 2026 entries end at June 2026.
THe difference in records, is caused by the removal of three Terminals from 2025 (ASARAMATORU (Ex Ima Terminal), ANAMBRA BASIN, UKPOKITI) and the addition of one in 2026 (CAWTHORNE).
Records which are not consistently included across all three datasets for 2024, 2025 and 2026 would be disregarded as this project seeks to analyze continuous production over the span of 2.5 years for each given Terminal.

In [11]:
df_2024.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
July               float64
August             float64
September          float64
October            float64
November           float64
December           float64
dtype: object

In [12]:
df_2025.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
July               float64
August             float64
September          float64
October            float64
November           float64
December           float64
dtype: object

In [13]:
df_2026.dtypes

Terminal/Stream        str
Liquid Type            str
January            float64
February           float64
March              float64
April              float64
May                float64
June               float64
dtype: object

Data types for each field are consistent with what is expected across the datasets.

In [40]:
null_2024_values = df_2024[df_2024[["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]].isnull().all(axis=1)]
null_2024_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,ASARAMATORU (Ex Ima Terminal),Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,ANAMBRA BASIN,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,UKPOKITI,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,IMA,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
null_2025_values = df_2025[df_2025[["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]].isnull().all(axis=1)]
null_2025_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,ASARAMATORU (Ex Ima Terminal),Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,ANAMBRA BASIN,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,UKPOKITI,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,IMA,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
null_2026_values = df_2026[df_2026[["January", "February", "March", "April", "May", "June"]].isnull().all(axis=1)]
null_2026_values

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
21,AJE,Crude Oil,NaN,NaN,NaN,NaN,NaN,NaN
22,AJE,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
23,AJE,Blend Total,NaN,NaN,NaN,NaN,NaN,NaN
47,IMA,Condensate,NaN,NaN,NaN,NaN,NaN,NaN


It can be seen that some records in each dataset are entirely empty.
Any records with completely missing values would be dropped during analysis in cleaning.
This is done to ensure ease and accuracy of analysis.

In [41]:
duplicates_2024 = df_2024[df_2024.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2024

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,TULJA - OKWUIBOME,Condensate,284045.0,264581.0,304064.0,299673.0,308497.0,280251.0,301720.0,283275.0,255131.0,296237.0,288743.0,282139.0


In [42]:
duplicates_2025 = df_2025[df_2025.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2025

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,TULJA - OKWUIBOME,Condensate,254417.0,272591.0,280584.0,267405.0,284381.0,276006.0,296915.0,303162.0,302246.0,320364.0,310201.0,318711.0


In [43]:
duplicates_2026 = df_2026[df_2026.duplicated(subset=["Terminal/Stream", "Liquid Type"], keep=False)]
duplicates_2026

,Terminal/Stream,Liquid Type,January,February,March,April,May,June
19,TULJA - OKWUIBOME,Condensate,NaN,NaN,NaN,NaN,NaN,NaN
49,TULJA - OKWUIBOME,Condensate,10.27,10.22,10.22,10.16,10.15,10.11


Seems to be a duplicate for the record of "TULJA - OKWUIBOME" Terminal in both 2025 and 2026 datasets. However, one of the entries is entirely null, so it would be discarded, and the entry with values would be used during analysis.

In [44]:
df_2024.describe()

,January,February,March,April,May,June,July,August,September,October,November,December
count,4.700000e+01,4.600000e+01,4.500000e+01,4.500000e+01,4.800000e+01,4.800000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.800000e+01,4.700000e+01,4.600000e+01
mean,3.886803e+06,3.464650e+06,3.558266e+06,3.459646e+06,3.364831e+06,3.333609e+06,3.587569e+06,3.767743e+06,3.511816e+06,3.513831e+06,3.873075e+06,4.018648e+06
std,9.598539e+06,8.439076e+06,8.487552e+06,8.387670e+06,8.388354e+06,8.297007e+06,8.857906e+06,9.219967e+06,8.656225e+06,8.843177e+06,9.612682e+06,9.940376e+06
min,6.413000e+03,4.265000e+03,6.954000e+03,6.427000e+03,3.896000e+03,6.970000e+03,6.789000e+03,5.246000e+03,5.821000e+03,6.910000e+02,5.462000e+03,5.498000e+03
25%,3.136330e+05,2.756445e+05,3.218740e+05,2.996730e+05,3.091998e+05,3.145505e+05,3.139880e+05,3.040902e+05,2.701315e+05,3.159208e+05,2.880235e+05,2.924245e+05
50%,9.453010e+05,1.036540e+06,1.061497e+06,8.892320e+05,8.322980e+05,8.149315e+05,9.281170e+05,9.379035e+05,1.044879e+06,1.108908e+06,9.546380e+05,1.185612e+06
75%,3.344956e+06,3.410196e+06,3.006372e+06,3.306128e+06,2.884685e+06,2.891586e+06,3.108878e+06,2.977762e+06,2.795426e+06,2.991664e+06,3.316910e+06,3.468504e+06
max,5.095380e+07,4.464866e+07,4.458200e+07,4.342305e+07,4.552568e+07,4.500597e+07,4.754465e+07,4.869112e+07,4.632869e+07,4.768198e+07,5.071454e+07,5.169436e+07


In [45]:
df_2025.describe()

,January,February,March,April,May,June,July,August,September,October,November,December
count,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.600000e+01,4.700000e+01,4.700000e+01
mean,4.113849e+06,3.565249e+06,3.767954e+06,3.868917e+06,3.912937e+06,3.893134e+06,4.057606e+06,3.939309e+06,3.614519e+06,3.867868e+06,3.703512e+06,3.728139e+06
std,1.023145e+07,8.844670e+06,9.364226e+06,9.607048e+06,9.703991e+06,9.684700e+06,1.006551e+07,9.664645e+06,8.971540e+06,9.501998e+06,9.225472e+06,9.337117e+06
min,5.934000e+03,4.508000e+03,5.221000e+03,4.865000e+03,5.043000e+03,4.438000e+03,4.741000e+03,4.741000e+03,4.590000e+03,4.666000e+03,4.565000e+03,4.616000e+03
25%,3.747460e+05,3.035360e+05,2.834890e+05,3.009920e+05,3.188700e+05,2.782285e+05,3.056510e+05,3.266940e+05,2.950640e+05,2.970438e+05,2.631515e+05,2.369545e+05
50%,1.050941e+06,1.080581e+06,1.145283e+06,1.051648e+06,1.178584e+06,1.161601e+06,1.189629e+06,1.251420e+06,1.152687e+06,1.294378e+06,1.212574e+06,1.119722e+06
75%,3.173460e+06,3.011052e+06,3.012108e+06,3.067994e+06,3.152703e+06,2.886904e+06,3.100470e+06,3.232327e+06,2.791990e+06,2.627290e+06,2.492530e+06,2.118758e+06
max,5.386188e+07,4.681470e+07,4.971706e+07,5.049921e+07,5.138048e+07,5.091136e+07,5.308074e+07,5.058000e+07,4.743544e+07,4.951318e+07,4.797161e+07,4.787469e+07


In [46]:
df_2026.describe()

,January,February,March,April,May,June
count,48.000000,47.000000,47.000000,48.000000,47.000000,47.000000
mean,121.997708,115.479362,118.110213,124.798750,130.962128,133.871064
std,308.584827,283.550812,294.975139,315.283581,326.805024,333.549098
min,0.120000,0.770000,0.780000,0.000000,0.090000,0.330000
25%,8.587500,8.435000,8.380000,7.795000,7.220000,7.355000
50%,40.450000,36.320000,41.570000,40.555000,41.720000,40.680000
75%,82.162500,67.165000,81.285000,76.657500,86.135000,88.860000
max,1627.460000,1483.950000,1546.090000,1663.410000,1700.800000,1735.400000


Significantly lower values in the description of the 2026 dataset is expected, as it was recorded in a different unit (thousand barrels per month). Conversion would be done in the cleaning phase.

In [47]:
df_2024.columns == df_2025.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [48]:
df_2025.columns[0:8] == df_2026.columns

array([ True,  True,  True,  True,  True,  True,  True,  True])

In [49]:
df_2024_cleaned = df_2024.drop(null_2024_values.index)
df_2025_cleaned = df_2025.drop(null_2025_values.index)
df_2026_cleaned = df_2026.drop(null_2026_values.index)

In [51]:
# Changing units of data in df_2026 from thousand_barrels_per_month to barrels_per_month
days_per_month = {
    "January": 31,
    "February": 28,
    "March": 31,
    "April": 30,
    "May": 31,
    "June": 30,
}

months = list(df_2026_cleaned.columns[2:])

for index, month in enumerate(months):
    df_2026_cleaned[month] = df_2026_cleaned.iloc[:, 2+index] * 1000 * days_per_month[month]
df_2026_cleaned


,Terminal/Stream,Liquid Type,January,February,March,April,May,June
0,BONNY,Crude Oil,7201300.0,7360640.0,7998000.0,8360400.0,8588240.0,9057900.0
1,BONNY,Condensate,389670.0,369040.0,427180.0,492900.0,521730.0,490200.0
2,BONNY,Blend Total,7590970.0,7729400.0,8424870.0,8853000.0,9110280.0,9548400.0
3,BRASS,Crude Oil,941160.0,963480.0,1087790.0,989700.0,1046870.0,995400.0
4,BRASS,Condensate,294810.0,281120.0,290470.0,255600.0,302870.0,260100.0
5,BRASS,Blend Total,1235970.0,1244600.0,1378260.0,1245000.0,1349740.0,1255500.0
6,QUA IBOE,Crude Oil,4549870.0,4401040.0,5196840.0,4918200.0,5371370.0,4892100.0
7,QUA IBOE,Condensate,57040.0,43680.0,49600.0,46800.0,2790.0,49500.0
8,QUA IBOE,Blend Total,4606910.0,4445000.0,5246440.0,4965000.0,5374160.0,4941600.0
9,FORCADOS,Crude Oil,7946850.0,6234760.0,4733080.0,6654000.0,8265530.0,8485800.0


In [52]:
df_2024_cleaned = round(df_2024_cleaned, 2)
df_2025_cleaned = round(df_2025_cleaned, 2)
df_2026_cleaned = round(df_2026_cleaned, 2)

In [53]:
df_2024_cleaned["Terminal/Stream"] = df_2024_cleaned["Terminal/Stream"].replace({"OYO": "OYO / OBODO", "OTAKPIPO (Ex Ima Terminal)": "OTAKPIPO"})
df_2024_cleaned

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,6351778.0,4604102.0,4260566.0,4158173.0,3605864.0,4536573.0,4584388.0,4894497.0,5589773.0,5765964.0,7051541.0,7274190.0
1,BONNY,Condensate,587495.0,583490.0,627057.0,543214.0,640026.0,568474.0,564610.0,584706.0,511021.0,494915.0,501329.0,509570.0
2,BONNY,Blend Total,6939273.0,5187592.0,4887623.0,4701387.0,4245890.0,5105047.0,5148998.0,5479203.0,6100794.0,6260879.0,7552870.0,7783759.0
3,BRASS,Crude Oil,735680.0,617189.0,686188.0,647053.0,635929.0,648849.0,664387.0,718104.0,842900.0,850352.0,775490.0,905544.0
4,BRASS,Condensate,160901.0,135498.0,158529.0,140724.0,153167.0,144677.0,148961.0,161633.0,201979.0,201451.0,179148.0,180417.0
5,BRASS,Blend Total,896581.0,752687.0,844717.0,787777.0,789096.0,793526.0,813348.0,879737.0,1044879.0,1051803.0,954638.0,1085961.0
6,QUA IBOE,Crude Oil,4251198.0,3664686.0,4058695.0,4284484.0,3962759.0,3261367.0,3343465.0,3073804.0,2833479.0,4114319.0,4171753.0,4132506.0
7,QUA IBOE,Condensate,45075.0,44684.0,37225.0,16091.0,3896.0,18795.0,18725.0,19933.0,18906.0,23269.0,20737.0,23315.0
8,QUA IBOE,Blend Total,4296273.0,3709370.0,4095920.0,4300575.0,3966655.0,3280162.0,3362190.0,3093737.0,2852385.0,4137588.0,4192490.0,4155821.0
9,FORCADOS,Crude Oil,7799819.0,6796892.0,6675119.0,6289085.0,5927475.0,6160694.0,6813681.0,8126915.0,6341130.0,4513085.0,7835499.0,7793585.0


In [56]:
df_2024_cleaned = df_2024_cleaned.drop(df_2024_cleaned[df_2024_cleaned["Liquid Type"] == "Condensate"].index)
df_2024_cleaned = df_2024_cleaned.drop(df_2024_cleaned[df_2024_cleaned["Liquid Type"] == "Blend Total"].index)
df_2024_cleaned = df_2024_cleaned.drop(df_2024_cleaned[df_2024_cleaned["Terminal/Stream"] == "TOTAL"].index)
df_2025_cleaned = df_2025_cleaned.drop(df_2025_cleaned[df_2025_cleaned["Liquid Type"] == "Condensate"].index)
df_2025_cleaned = df_2025_cleaned.drop(df_2025_cleaned[df_2025_cleaned["Liquid Type"] == "Blend Total"].index)
df_2025_cleaned = df_2025_cleaned.drop(df_2025_cleaned[df_2025_cleaned["Terminal/Stream"] == "TOTAL"].index)
df_2026_cleaned = df_2026_cleaned.drop(df_2026_cleaned[df_2026_cleaned["Liquid Type"] == "Condensate"].index)
df_2026_cleaned = df_2026_cleaned.drop(df_2026_cleaned[df_2026_cleaned["Liquid Type"] == "Blend Total"].index)
df_2026_cleaned = df_2026_cleaned.drop(df_2026_cleaned[df_2026_cleaned["Terminal/Stream"] == "TOTAL"].index)

In [55]:
df_2025_cleaned

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,7570173.0,5892185.0,7360738.0,7079233.0,6673050.0,6625912.0,7567542.0,5850190.0,5924385.0,7535315.0,7751499.0,7884015.0
3,BRASS,Crude Oil,860312.0,726912.0,937437.0,619261.0,857711.0,710231.0,820821.0,846393.0,935010.0,946439.0,981254.0,914165.0
6,QUA IBOE,Crude Oil,4580997.0,4227836.0,3504376.0,4296506.0,4921690.0,5036322.0,4503690.0,4946259.0,4640056.0,2683665.0,3980764.0,5074619.0
9,FORCADOS,Crude Oil,7997419.0,6919400.0,5798465.0,8409659.0,7089537.0,7901129.0,8059799.0,8084121.0,6853852.0,8585208.0,8327198.0,8569954.0
12,ESCRAVOS (Oil Terminal),Crude Oil,4261436.0,3794400.0,4186992.0,4082589.0,4439405.0,4047894.0,4335432.0,4080954.0,4129694.0,4329068.0,3829825.0,4091466.0
15,ODUDU (AMENAM BLEND),Crude Oil,2324122.0,2063093.0,2313424.0,2106782.0,1890759.0,2061295.0,2124883.0,1846043.0,2005854.0,2111519.0,1888702.0,1970777.0
18,TULJA - OKWUIBOME,Crude Oil,2260611.0,1899107.0,2110574.0,2065631.0,2119797.0,2024726.0,2080781.0,2009287.0,1627130.0,1581339.0,1591292.0,1561674.0
24,OKORO (Ex Ima Terminal),Crude Oil,168432.0,141970.0,160308.0,158837.0,154386.0,143657.0,137526.0,72429.0,134175.0,147289.0,145562.0,163212.0
26,OTAKPIPO,Crude Oil,210114.0,125233.0,114411.0,148290.0,212252.0,220666.0,185672.0,215945.0,206713.0,204048.0,194965.0,199316.0
27,ANTAN,Crude Oil,264704.0,233546.0,257473.0,167148.0,177763.0,198236.0,161708.0,151628.0,145910.0,167509.0,29839.0,136740.0


In [57]:
df_2024_cleaned

,Terminal/Stream,Liquid Type,January,February,March,April,May,June,July,August,September,October,November,December
0,BONNY,Crude Oil,6351778.0,4604102.0,4260566.0,4158173.0,3605864.0,4536573.0,4584388.0,4894497.0,5589773.0,5765964.0,7051541.0,7274190.0
3,BRASS,Crude Oil,735680.0,617189.0,686188.0,647053.0,635929.0,648849.0,664387.0,718104.0,842900.0,850352.0,775490.0,905544.0
6,QUA IBOE,Crude Oil,4251198.0,3664686.0,4058695.0,4284484.0,3962759.0,3261367.0,3343465.0,3073804.0,2833479.0,4114319.0,4171753.0,4132506.0
9,FORCADOS,Crude Oil,7799819.0,6796892.0,6675119.0,6289085.0,5927475.0,6160694.0,6813681.0,8126915.0,6341130.0,4513085.0,7835499.0,7793585.0
12,ESCRAVOS (Oil Terminal),Crude Oil,4178516.0,3684339.0,4012721.0,3866374.0,4113862.0,3996696.0,4385991.0,4209219.0,4080483.0,4276766.0,3992925.0,3807346.0
15,ODUDU (AMENAM BLEND),Crude Oil,2934994.0,2722999.0,2836178.0,2659792.0,2630817.0,2572785.0,2468442.0,2583858.0,2462691.0,2526821.0,2423720.0,2446481.0
18,TULJA - OKWUIBOME,Crude Oil,1762863.0,1732677.0,1859018.0,1837364.0,1972551.0,1928150.0,1992021.0,2008090.0,2029787.0,2212884.0,2219874.0,2250725.0
24,OKORO (Ex Ima Terminal),Crude Oil,203761.0,190965.0,201924.0,182645.0,188990.0,167626.0,181758.0,173385.0,167614.0,177386.0,172483.0,173580.0
26,OTAKPIPO,Crude Oil,278843.0,217705.0,242882.0,270168.0,306424.0,270141.0,266605.0,197152.0,195878.0,169735.0,127545.0,186192.0
27,ANTAN,Crude Oil,348555.0,339797.0,351219.0,316850.0,347468.0,338999.0,326256.0,301795.0,282378.0,322482.0,262803.0,257832.0


In [64]:
df_2024_cleaned["Terminal/Stream"] == df_2025_cleaned["Terminal/Stream"]

0     True
3     True
6     True
9     True
12    True
15    True
18    True
24    True
26    True
27    True
28    True
29    True
30    True
31    True
32    True
34    True
35    True
36    True
37    True
38    True
39    True
40    True
42    True
43    True
44    True
45    True
46    True
Name: Terminal/Stream, dtype: bool

In [ ]:
terminals_removed_from_2025 = [terminal for terminal in list(df_2025_cleaned["Terminal/Stream"]) if terminal not in list(df_2026_cleaned["Terminal/Stream"])]
terminals_added_to_2026 = [terminal for terminal in list(df_2026_cleaned["Terminal/Stream"]) if terminal not in list(df_2025_cleaned["Terminal/Stream"])]

print(f"Terminals removed from 2025 in 2026: {terminals_removed_from_2025}")
print(f"Terminals added to 2026 from 2025: {terminals_added_to_2026}")


Terminals removed from 2025 in 2026: []
Terminals added to 2026 from 2025: ['CAWTHORNE']


The exact same terminals from 2024 carried on through the cleaning process into 2025. However, in 2026, a new terminal was added: CAWTHORNE.

In [69]:
df_2024_cleaned.set_index("Terminal/Stream").to_csv("../data/cleaned/NUPRC_2024_production_cleaned.csv")
df_2025_cleaned.set_index("Terminal/Stream").to_csv("../data/cleaned/NUPRC_2025_production_cleaned.csv")
df_2026_cleaned.set_index("Terminal/Stream").to_csv("../data/cleaned/NUPRC_2026_production_cleaned.csv")